# Clase 6 · En vivo: un equipo de agentes, con firma

Este cuaderno lo corremos **juntos en clase**: cada sección es una slide de hoy (M7). Antes de correr cada celda,
**anotá tu predicción**; después comparamos. Si te da distinto que en la slide, también es un resultado.

- Corré las celdas en orden con **Shift + Enter**. La clave: **Secretos** (la llave de la izquierda) → `GOOGLE_API_KEY`.
- Gasta unos 50 pedidos de los 500 del día; 30 son de la sección 1, que es opcional. Las secciones 4 y 5 no gastan.
- La web acá es **grabada** (siempre la misma respuesta) o **caída** (devuelve un ERROR): así se repite igual en clase.
- **No es tu práctica.** Este cuaderno es para mirar y predecir: todo viene escrito. Tu práctica (M7) la escribís vos, en tu repo o en su propio cuaderno de Colab (el botón de cada práctica en ucu-la).

<details><summary><b>¿Sale un error?</b></summary>

- **«No pude leer el secreto GOOGLE_API_KEY»** o **`API key not valid`:** revisá el nombre del secreto (exacto, en mayúsculas),
  que «Acceso del notebook» esté activado y que la clave esté bien copiada.
- **`503 UNAVAILABLE`** («high demand»): el modelo está saturado, no es tu código. Esperá un minuto y volvé a correr.
- **`429`** (*quota*, *RESOURCE_EXHAUSTED*): se terminó la cuota gratis del día o del minuto. Esperá o seguí mañana.
- **`NameError`:** salteaste una celda o Colab se reinició. Menú *Entorno de ejecución* → *Run before*.
</details>

## 0 · Preparar (una vez, un par de minutos)

In [ ]:
%pip install -q langchain==1.4.6 langchain-google-genai==4.4.2 chromadb==1.5.9 langchain-text-splitters==1.1.3
import os

try:
    from google.colab import userdata                      # en Colab: la clave sale de los Secretos
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except ImportError:
    pass                                                   # fuera de Colab: GOOGLE_API_KEY ya tiene que estar
except Exception as error:                                 # no hay secreto con ese nombre o le falta el acceso
    print("No pude leer el secreto GOOGLE_API_KEY:", type(error).__name__)

from langchain.chat_models import init_chat_model

modelo = init_chat_model("google_genai:gemini-3.5-flash-lite", max_output_tokens=300)  # tope de salida: cuida la cuota
print("Listo." if os.environ.get("GOOGLE_API_KEY") else "Falta la clave: Secretos → GOOGLE_API_KEY.")

In [ ]:
import json
import time
import urllib.request
import uuid
from pathlib import Path
from typing import Literal

import chromadb
import pandas as pd
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware
from langchain.tools import tool
from langchain_core.callbacks import BaseCallbackHandler
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver

pd.set_option("display.max_colwidth", 80)
# los 5 documentos de ejemplo, las 12 preguntas de selección y la web grabada, de la plantilla del proyecto
PLANTILLA = "https://raw.githubusercontent.com/JuanFKurucz/ucu-la-project/main/"
for archivo in ["corpus/langchain__short-term-memory.md", "corpus/langgraph__interrupts.md", "corpus/langgraph__persistence.md",
                "corpus/langgraph__thinking-in-langgraph.md", "corpus/langgraph__use-subgraphs.md",
                "datos/m6_seleccion.jsonl", "fixtures/web.json"]:
    Path(archivo).parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve(PLANTILLA + archivo, archivo)
seleccion = [json.loads(linea) for linea in open("datos/m6_seleccion.jsonl", encoding="utf-8") if linea.strip()]

cortador = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
coleccion = chromadb.Client().get_or_create_collection("corpus_1000_150")
for archivo in sorted(Path("corpus").glob("*.md")):
    partes = cortador.split_text(archivo.read_text(encoding="utf-8"))
    coleccion.upsert(ids=[f"{archivo.stem}#{i}" for i in range(len(partes))], documents=partes,
                     metadatas=[{"doc": archivo.stem}] * len(partes))
os.environ["WEB"] = "grabada"  # "grabada": siempre la misma respuesta · "caida": devuelve un ERROR


@tool
def buscar_documentos(consulta: str) -> str:
    """Busca en la documentación del corpus (LangGraph/LangChain: persistencia, interrupts,
    subgrafos, memoria). Usar para preguntas técnicas sobre esos temas."""
    r = coleccion.query(query_texts=[consulta], n_results=3)
    return "\n\n".join(f"[{m['doc']}] {t[:600]}" for m, t in zip(r["metadatas"][0], r["documents"][0]))


@tool
def buscar_web(consulta: str) -> str:
    """Busca en internet información reciente o que no está en la documentación del corpus
    (noticias, precios, versiones nuevas, empresas, fechas)."""
    if os.environ["WEB"] == "caida":
        return "ERROR: la búsqueda web no respondió. No inventes; decíselo al usuario."
    return json.loads(Path("fixtures/web.json").read_text(encoding="utf-8"))["resultado"]


@tool
def calcular(operacion: Literal["suma", "resta", "multiplicacion", "division", "porcentaje_cambio"],
             a: float, b: float) -> str:
    """Hace una cuenta con dos números. porcentaje_cambio = (b - a) / a * 100. Usar para cualquier cuenta."""
    resultado = {"suma": a + b, "resta": a - b, "multiplicacion": a * b, "division": a / b if b else None,
                 "porcentaje_cambio": (b - a) / a * 100 if a else None}[operacion]
    return "ERROR: división por cero. No inventes el resultado." if resultado is None else f"{round(resultado, 4) + 0.0:.12g}"


SISTEMA = ("Sos un asistente de investigación. Usá buscar_documentos para LangGraph/LangChain, "
           "buscar_web para información reciente o externa y calcular para cualquier cuenta. "
           "Si una herramienta devuelve ERROR, decilo y no inventes. Respondé en español, breve.")


def topes():  # los límites van en el código, no en el prompt
    return [ModelCallLimitMiddleware(run_limit=6), ToolCallLimitMiddleware(run_limit=5)]


agente = create_agent(modelo, tools=[buscar_documentos, buscar_web, calcular], system_prompt=SISTEMA,
                      middleware=topes(), checkpointer=InMemorySaver())  # tu v0.4


class Contador(BaseCallbackHandler):
    """Cuenta las llamadas al modelo: LangChain avisa cada vez que empieza una."""
    def __init__(self):
        self.llamadas = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.llamadas += 1


def preguntar(pregunta):
    contador = Contador()
    config = {"configurable": {"thread_id": uuid.uuid4().hex}, "callbacks": [contador]}
    mensajes = agente.invoke({"messages": [("user", pregunta)]}, config)["messages"]
    return mensajes, contador.llamadas


def pedidas(mensajes):
    return [p["name"] for m in mensajes for p in getattr(m, "tool_calls", [])]


def mostrar(mensajes):
    for m in mensajes:
        if m.type == "human":
            print(f"[pregunta]     {m.text}")
        elif m.type == "ai" and m.tool_calls:
            for pedido in m.tool_calls:
                print(f"[modelo pide]  {pedido['name']}({pedido['args']})")
        elif m.type == "tool":
            print(f"[tu código]    {m.name} → {m.text[:90]!r}")
        else:
            print(f"[respuesta]    {m.text}")


print(f"{coleccion.count()} fragmentos · {len(seleccion)} preguntas de selección · tu v0.4 lista")

# De un agente a un equipo: el grafo

## 1 · Un grafo: un diagrama de flujo que corre (M7, slides 7 y 10) · sin gastar

Un grafo de LangGraph, **sin modelo**, para ver las piezas: cada **nodo** es una función de Python, cada **arista** dice
qué sigue (una **condicional** la decide una función) y el **estado** es una fila que pasa de caja en caja.
Cada nodo devuelve **solo lo que cambia**. **Predecí:** con «¿Cuánto es 15 % de 200?», ¿por qué cajas pasa?

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph


class Fila(TypedDict, total=False):
    pregunta: str
    tipo: str
    notas: Annotated[list[str], operator.add]  # reducer: cada nodo SUMA notas, no las pisa
    respuesta: str


def clasificar(fila):
    tipo = "cuenta" if any(c.isdigit() for c in fila["pregunta"]) else "documentos"
    print(f"▶ clasificar · recibe «{fila['pregunta']}» · devuelve tipo = {tipo}")
    return {"tipo": tipo}


def contar(fila):
    print("▶ contar      · suma una nota")
    return {"notas": ["15 % de 200 = 30"]}


def buscar(fila):
    print("▶ buscar      · suma una nota")
    return {"notas": [buscar_documentos.invoke({"consulta": fila["pregunta"]})[:80] + "…"]}


def responder(fila):
    print(f"▶ responder   · recibe {len(fila['notas'])} nota(s)")
    return {"respuesta": " · ".join(fila["notas"])}


g = StateGraph(Fila)
for nombre, funcion in [("clasificar", clasificar), ("contar", contar), ("buscar", buscar), ("responder", responder)]:
    g.add_node(nombre, funcion)
g.add_edge(START, "clasificar")
g.add_conditional_edges("clasificar", lambda fila: fila["tipo"], {"cuenta": "contar", "documentos": "buscar"})
g.add_edge("contar", "responder")
g.add_edge("buscar", "responder")
g.add_edge("responder", END)
flujo = g.compile()

print(flujo.invoke({"pregunta": "¿Cuánto es 15 % de 200?"}), "\n")
print(flujo.invoke({"pregunta": "¿Qué guarda un checkpointer?"}), "\n")
print(flujo.get_graph().draw_mermaid())  # copialo en https://mermaid.live

**Mirá:** la flecha de `clasificar` es **condicional**: en el dibujo de mermaid.live sale punteada. El estado final tiene
todo lo que fue completando cada caja, como una fila con columnas (pregunta, tipo, notas, respuesta).

## 2 · La persona que aprueba: pausa y retoma (M7, slides 12 y 15) · sin gastar

`interrupt()` **pausa** el grafo y el **checkpointer** guarda el estado con su hilo (`thread_id`). Después,
`Command(resume=…)` retoma **el mismo hilo** con la decisión de la persona. **Predecí:** ¿qué devuelve el grafo en la pausa?

In [ ]:
from langgraph.types import Command, interrupt


class Pedido(TypedDict, total=False):
    borrador: str
    aprobada: bool
    respuesta_final: str


def aprobar(pedido):
    decision = interrupt({"borrador": pedido["borrador"]})  # PAUSA: el estado queda guardado
    return {"aprobada": decision is True}                   # cualquier otra cosa no aprueba


g = StateGraph(Pedido)
g.add_node("aprobar", aprobar)
g.add_node("final", lambda pedido: {"respuesta_final": pedido["borrador"]})
g.add_edge(START, "aprobar")
g.add_conditional_edges("aprobar", lambda p: "final" if p["aprobada"] else END, {"final": "final", END: END})
g.add_edge("final", END)
firma = g.compile(checkpointer=InMemorySaver())

for hilo, decision in (("aprueba", True), ("rechaza", False)):
    config = {"configurable": {"thread_id": hilo}}
    pausa = firma.invoke({"borrador": "La tabla tiene 5 filas; × 3 = 15."}, config)
    print(f"hilo {hilo} · en pausa · le toca a {firma.get_state(config).next} · {pausa['__interrupt__'][0].value}")
    final = firma.invoke(Command(resume=decision), config)  # mismo hilo = mismo estado
    print(f"hilo {hilo} · respuesta final: {final.get('respuesta_final', '(ninguna: el rechazo la bloqueó)')}\n")

**Mirá:** en la pausa el grafo no terminó: `next` dice que le toca a `aprobar`. Si cerrás Colab entre la pausa y la
decisión, con `InMemorySaver` se pierde; con una base (Postgres) seguiría ahí.

## 3 · Tu v0.5: un supervisor, tres especialistas, una firma (M7, slides 4, 9 y 13) · gasta unos 8 pedidos

Un grafo como el de tu práctica M7: el **supervisor** (con el modelo) elige quién sigue, el **investigador**
busca, el **analista** calcula, el **redactor** escribe solo con las notas y una persona aprueba.
**Predecí:** ¿cuántas llamadas al modelo hace el equipo, contra el agente solo?

**Una diferencia con tu práctica M7:** acá el supervisor elige con un **molde** (`with_structured_output(Ruta)`): no puede contestar otra cosa que los tres nombres. En la práctica contesta una palabra y un `CASE WHEN` la lee. Las dos sirven.

In [ ]:
from pydantic import BaseModel

INVESTIGADOR = ("Investigás: buscar_documentos para LangGraph/LangChain, buscar_web para datos externos "
                "o recientes. Devolvé hallazgos breves con su fuente. Si una herramienta da ERROR, decilo.")
SUPERVISOR = ("Elegí el próximo paso. investigador: falta información. analista: hay que calcular "
              "con lo hallado. redactor: ya alcanza.")
investigador = create_agent(modelo, tools=[buscar_documentos, buscar_web], system_prompt=INVESTIGADOR, middleware=topes())
analista = create_agent(modelo, tools=[calcular], system_prompt="Hacés cuentas con calcular sobre los hallazgos. Breve.",
                        middleware=topes())


class Ruta(BaseModel):
    """Molde para la decisión del supervisor: solo puede elegir una de estas tres."""
    siguiente: Literal["investigador", "analista", "redactor"]


class Estado(TypedDict, total=False):
    pregunta: str
    notas: Annotated[list[str], operator.add]
    pasos: int
    siguiente: str
    borrador: str
    aprobada: bool
    respuesta_final: str


def supervisor(estado):
    if estado.get("pasos", 0) >= 3:  # tope en código: no depende del modelo
        return {"siguiente": "redactor"}
    ruta = modelo.with_structured_output(Ruta).invoke(
        f"{SUPERVISOR}\nPregunta: {estado['pregunta']}\nNotas: {estado.get('notas') or 'ninguna'}")
    siguiente = ruta.siguiente if ruta else "redactor"
    print(f"▶ supervisor elige: {siguiente}")
    return {"siguiente": siguiente, "pasos": estado.get("pasos", 0) + 1}


def especialista(nombre, agente_):
    def nodo(estado):
        mensajes = agente_.invoke({"messages": [("user", f"{estado['pregunta']}\nNotas previas: {estado.get('notas') or 'ninguna'}")]})["messages"]
        print(f"▶ {nombre} pidió {pedidas(mensajes)} y anotó: {mensajes[-1].text[:80]!r}")
        return {"notas": [mensajes[-1].text]}
    return nodo


def redactor(estado):
    texto = modelo.invoke(f"Respondé '{estado['pregunta']}' en 2 o 3 frases, SOLO con estas notas; "
                          f"si no alcanzan, decilo.\nNotas: {estado.get('notas') or 'ninguna'}").text
    return {"borrador": texto}


g = StateGraph(Estado)
for nombre, funcion in [("supervisor", supervisor), ("investigador", especialista("investigador", investigador)),
                        ("analista", especialista("analista", analista)), ("redactor", redactor), ("aprobar", aprobar),
                        ("final", lambda e: {"respuesta_final": e["borrador"]})]:
    g.add_node(nombre, funcion)
g.add_edge(START, "supervisor")
g.add_conditional_edges("supervisor", lambda e: e["siguiente"],
                        {"investigador": "investigador", "analista": "analista", "redactor": "redactor"})
g.add_edge("investigador", "supervisor")
g.add_edge("analista", "supervisor")
g.add_edge("redactor", "aprobar")
g.add_conditional_edges("aprobar", lambda e: "final" if e["aprobada"] else END, {"final": "final", END: END})
g.add_edge("final", END)
equipo = g.compile(checkpointer=InMemorySaver())

PREGUNTA = "¿Qué hace interrupt() en LangGraph?"  # cambiala por una que dicte la clase
contador = Contador()
config = {"configurable": {"thread_id": uuid.uuid4().hex}, "callbacks": [contador]}
pausa = equipo.invoke({"pregunta": PREGUNTA}, config)
print("\nBorrador para aprobar:", pausa["borrador"])
final = equipo.invoke(Command(resume=True), config)  # la persona aprueba (probá con False)
_, solo = preguntar(PREGUNTA)
print(f"\nRespuesta final: {final.get('respuesta_final')}\nLlamadas al modelo · equipo: {contador.llamadas} · agente solo: {solo}")

**Mirá:** el equipo gasta varias veces las llamadas del agente solo (slides 4 y 5: en la suite del curso, 2,6 veces y
peor elección). Más agentes no es mejor por sí solo: se justifica si lo **medís**. El dibujo de este equipo sale con
`print(equipo.get_graph().draw_mermaid())`; pegalo en mermaid.live (taller de hoy).

# El equipo con una persona que firma

## 4 · El equipo, paso a paso (M7, slides 13 y 14)

Tu v0.5: el supervisor elige quién sigue, el investigador busca, el analista calcula, el redactor escribe y una persona
firma. Acá cada caja imprime lo que **devuelve**, como en la slide 16. **Predecí:** con la pregunta de la tabla × 3,
¿el supervisor llama al analista para la cuenta?

**Una diferencia con tu práctica M7:** acá el supervisor elige con un **molde** (`with_structured_output(Ruta)`): no puede contestar otra cosa que los tres nombres. En la práctica contesta una palabra y un `CASE WHEN` la lee. Las dos sirven.

In [ ]:
import operator
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt
from pydantic import BaseModel

INVESTIGADOR = ("Investigás: buscar_documentos para LangGraph/LangChain, buscar_web para datos externos "
                "o recientes. Devolvé hallazgos breves con su fuente. Si una herramienta da ERROR, decilo.")
SUPERVISOR = ("Elegí el próximo paso. investigador: falta información. analista: hay que calcular "
              "con lo hallado. redactor: ya alcanza.")


class Ruta(BaseModel):
    """Molde para la decisión del supervisor: solo puede elegir una de estas tres."""
    siguiente: Literal["investigador", "analista", "redactor"]


class Estado(TypedDict, total=False):
    pregunta: str
    notas: Annotated[list[str], operator.add]  # cada especialista SUMA notas, no las pisa
    pasos: int
    siguiente: str
    borrador: str
    aprobada: bool
    respuesta_final: str


def armar_equipo(instrucciones_supervisor):
    investigador = create_agent(modelo, tools=[buscar_documentos, buscar_web], system_prompt=INVESTIGADOR, middleware=topes())
    analista = create_agent(modelo, tools=[calcular], system_prompt="Hacés cuentas con calcular sobre los hallazgos. Breve.",
                            middleware=topes())

    def supervisor(estado):
        if estado.get("pasos", 0) >= 3:  # tope en código: no depende del modelo
            return {"siguiente": "redactor"}
        ruta = modelo.with_structured_output(Ruta).invoke(
            f"{instrucciones_supervisor}\nPregunta: {estado['pregunta']}\nNotas: {estado.get('notas') or 'ninguna'}")
        return {"siguiente": ruta.siguiente if ruta else "redactor", "pasos": estado.get("pasos", 0) + 1}

    def especialista(agente_):
        def nodo(estado):
            mensajes = agente_.invoke({"messages": [("user", f"{estado['pregunta']}\nNotas previas: {estado.get('notas') or 'ninguna'}")]})["messages"]
            return {"notas": [mensajes[-1].text]}
        return nodo

    def redactor(estado):
        return {"borrador": modelo.invoke(f"Respondé '{estado['pregunta']}' en 2 o 3 frases, SOLO con estas notas; "
                                          f"si no alcanzan, decilo.\nNotas: {estado.get('notas') or 'ninguna'}").text}

    def aprobar(estado):
        return {"aprobada": interrupt({"borrador": estado["borrador"]}) is True}  # PAUSA hasta que una persona decide

    g = StateGraph(Estado)
    for nombre, funcion in [("supervisor", supervisor), ("investigador", especialista(investigador)),
                            ("analista", especialista(analista)), ("redactor", redactor), ("aprobar", aprobar),
                            ("final", lambda e: {"respuesta_final": e["borrador"]})]:
        g.add_node(nombre, funcion)
    g.add_edge(START, "supervisor")
    g.add_conditional_edges("supervisor", lambda e: e["siguiente"],
                            {"investigador": "investigador", "analista": "analista", "redactor": "redactor"})
    g.add_edge("investigador", "supervisor")
    g.add_edge("analista", "supervisor")
    g.add_edge("redactor", "aprobar")
    g.add_conditional_edges("aprobar", lambda e: "final" if e["aprobada"] else END, {"final": "final", END: END})
    g.add_edge("final", END)
    return g.compile(checkpointer=InMemorySaver())


def correr_hasta_la_firma(equipo, pregunta):
    """Corre el equipo hasta la pausa e imprime lo que devuelve cada caja; devuelve el hilo y la ruta."""
    hilo = {"configurable": {"thread_id": uuid.uuid4().hex}}
    ruta = []
    for paso in equipo.stream({"pregunta": pregunta}, hilo, stream_mode="updates"):
        for caja, cambio in paso.items():
            if caja == "__interrupt__":
                print(f"[PAUSA]        espera a una persona · borrador: {cambio[0].value['borrador'][:110]!r}")
            else:
                ruta.append(caja)
                print(f"[{caja}]".ljust(15), str(cambio)[:110])
    return hilo, ruta


PREGUNTA = ("Según la documentación, ¿cuántas filas tiene la tabla que compara checkpointer y store? "
            "Multiplicá ese número por 3.")
equipo = armar_equipo(SUPERVISOR)
hilo, ruta = correr_hasta_la_firma(equipo, PREGUNTA)
print("\nLa ruta:", " → ".join(ruta), "· ¿llamó al analista?", "analista" in ruta)

**Mirá:** si el analista no aparece en la ruta, la cuenta la hizo el redactor **sin calculadora**: puede salir bien,
pero nadie la comprobó (slide 16). ¿Cómo harías que pida al analista cuando hay una cuenta?

## 5 · El grafo se pausa y retoma (M7, slide 15) · sin gastar más

El grafo quedó **en pausa** en `aprobar`. El checkpointer guarda el estado del hilo, como el paso «Iniciar y esperar
una aprobación» de Power Automate. **Predecí:** ¿qué hay guardado? Después decidí vos: `True` aprueba, `False` rechaza.

In [ ]:
pausa = equipo.get_state(hilo)
print("Le toca a:", pausa.next, "· guardado:", list(pausa.values), "\n")

DECISION = True  # tu firma: True aprueba, False rechaza (con False, volvé a correr la sección 1 para otro hilo)
final = equipo.invoke(Command(resume=DECISION), hilo)  # mismo hilo = mismo estado
print("Respuesta final:", final.get("respuesta_final", "(ninguna: el rechazo la bloqueó)"))

**Mirá:** `Le toca a: ('aprobar',)`: el grafo espera tu firma con el estado guardado. Con `True` sale la respuesta; con `False`, ninguna: el rechazo la bloquea en código.

## 6 · ¿Qué cambiarías en el supervisor? (M7, slide 17) · gasta unos 6 pedidos, opcional

Tres perillas: el **modelo**, las **instrucciones** o el **tope de pasos**. Acá cambiamos solo las instrucciones.
**Predecí:** si le decís que toda cuenta va al analista, ¿lo llama?

In [ ]:
OTRAS = SUPERVISOR + " Si la pregunta pide una cuenta, el analista la hace antes del redactor."  # cambialas vos
hilo2, ruta2 = correr_hasta_la_firma(armar_equipo(OTRAS), PREGUNTA)
print("\nAntes:", " → ".join(ruta), "\nAhora:", " → ".join(ruta2))

**Mirá:** cambiaste **una sola cosa**. Para saber si mejoró de verdad, la medida es la suite entera (práctica M7), no una
pregunta.
